# TP — Part 1: Data Profiling
### Dataset: Road-traffic personal-injury accidents (BAAC 2024) — data.gouv.fr


In [1]:
import pandas as pd
import numpy as np
PATH = "."

files = {
    "caract":    "caract-2024.csv",
    "lieux":     "lieux-2024.csv",
    "vehicules": "vehicules-2024.csv",
    "usagers":   "usagers-2024.csv",
}

raw = {
    name: pd.read_csv(f"{PATH}/{fname}", sep=";", dtype=str, encoding="utf-8")
    for name, fname in files.items()
}

for name, df in raw.items():
    print(f"{name:10s} : {df.shape[0]:>7} lignes × {df.shape[1]} colonnes")

caract     :   54402 lignes × 15 colonnes
lieux      :   70248 lignes × 18 colonnes
vehicules  :   92678 lignes × 11 colonnes
usagers    :  125187 lignes × 16 colonnes


In [4]:
for name, df in raw.items():
    print(f"\n===== {name} ({df.shape[1]} colonnes) =====")
    inv = pd.DataFrame({
        "colonne": df.columns,
        "dtype_brut": [str(df[c].dtype) for c in df.columns]
    })
    print(inv.to_string(index=False))


===== caract (15 colonnes) =====
colonne dtype_brut
Num_Acc     object
   jour     object
   mois     object
     an     object
   hrmn     object
    lum     object
    dep     object
    com     object
    agg     object
    int     object
    atm     object
    col     object
    adr     object
    lat     object
   long     object

===== lieux (18 colonnes) =====
colonne dtype_brut
Num_Acc     object
   catr     object
   voie     object
     v1     object
     v2     object
   circ     object
    nbv     object
   vosp     object
   prof     object
     pr     object
    pr1     object
   plan     object
 lartpc     object
larrout     object
   surf     object
  infra     object
   situ     object
    vma     object

===== vehicules (11 colonnes) =====
    colonne dtype_brut
    Num_Acc     object
id_vehicule     object
    num_veh     object
       senc     object
       catv     object
        obs     object
       obsm     object
       choc     object
       manv     object
 

###  Meaning of the columns

**`caract` — accident characteristics** (1 row / accident)
- `Num_Acc`: unique accident identifier (primary key of the dataset)
- `jour`, `mois`, `an`, `hrmn`: date and time of the accident
- `lum`: lighting conditions (1 daylight · 2 dusk/dawn · 3 night no lighting · 4 night lighting off · 5 night lighting on)
- `dep`, `com`: department and municipality (INSEE code)
- `agg`: location (1 outside built-up area · 2 in built-up area)
- `int`: intersection type
- `atm`: weather conditions (1 normal · 2 light rain · … · 9 other)
- `col`: collision type
- `adr`: postal address (free text)
- `lat`, `long`: GPS coordinates

**`lieux` — location description** (1 row / road segment)
- `catr`: road category (1 motorway · 2 national · 3 departmental · 4 municipal · …)
- `voie`, `v1`, `v2`: road identification
- `circ`: traffic regime · `nbv`: number of lanes · `vosp`: reserved lane
- `prof`: longitudinal profile (gradient) · `plan`: layout (straight / curve)
- `pr`, `pr1`: reference markers (kilometre points)
- `lartpc`, `larrout`: widths (central reservation, carriageway)
- `surf`: surface condition (1 normal · 2 wet · …) · `infra`: infrastructure · `situ`: situation
- `vma`: maximum authorised speed (km/h)

**`vehicules` — vehicles involved** (1 row / vehicle)
- `id_vehicule`, `num_veh`: vehicle identifiers within the accident
- `senc`: direction · `catv`: vehicle category · `obs` / `obsm`: fixed / moving obstacle hit
- `choc`: initial point of impact · `manv`: manoeuvre · `motor`: engine type · `occutc`: occupants (public transport)

**`usagers` — people involved** (1 row / person)
- `place`: seat position in the vehicle
- `catu`: user category (**1 driver · 2 passenger · 3 pedestrian**)
- **`grav`: injury severity (1 unharmed · 2 killed · 3 hospitalised · 4 slight injury)** ← target variable for severity analysis
- `sexe`: sex (1 male · 2 female) · `an_nais`: birth year · `trajet`: trip purpose
- `secu1`, `secu2`, `secu3`: safety equipment used
- `locp`, `actp`, `etatp`: pedestrian location / action / state

In [5]:
SENTINELS = {"", " ", "-1", "N/A", "n/a", "nan", "NaN", "#N/A", "null", "NULL"}

def normalize_missing(s: pd.Series) -> pd.Series:
    s2 = s.astype("string").str.strip()
    return s2.mask(s2.isin(SENTINELS))

def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    d = df.apply(normalize_missing)
    n_miss = d.isna().sum()
    pct = (n_miss / len(d) * 100).round(2)
    return (pd.DataFrame({"n_missing": n_miss, "pct_missing": pct})
              .sort_values("pct_missing", ascending=False))

for name, df in raw.items():
    print(f"\n===== {name} — taux de manquants (sentinelles incluses) =====")
    print(missing_report(df).to_string())


===== caract — taux de manquants (sentinelles incluses) =====
         n_missing  pct_missing
adr           2310         4.25
col              6         0.01
Num_Acc          0         0.00
an               0         0.00
hrmn             0         0.00
jour             0         0.00
mois             0         0.00
dep              0         0.00
lum              0         0.00
com              0         0.00
agg              0         0.00
atm              0         0.00
int              0         0.00
lat              0         0.00
long             0         0.00

===== lieux — taux de manquants (sentinelles incluses) =====
         n_missing  pct_missing
lartpc       70215        99.95
v2           64332        91.58
larrout      48646        69.25
pr1          27432        39.05
pr           27364        38.95
v1           16272        23.16
voie         13331        18.98
circ          4354         6.20
nbv           4178         5.95
vosp          3832         5.45
vma        

### Critical missingness

We must distinguish **expected structural missingness** from **critical missingness** :

| Column | Table | ~% missing | Nature | Criticality |
|---|---|---|---|---|
| `lartpc` | lieux | ≈ 100 % | central-reservation width: exists only on divided roads | **Low** (structural) |
| `occutc` | vehicules | ≈ 99 % | public-transport occupants: empty except buses/coaches | **Low** (structural) |
| `etatp`, `secu3`, `locp`, `actp` | usagers | 49–92 % | pedestrian-specific fields | **Low** (structural) |
| `v2`, `larrout`, `pr`, `pr1`, `v1` | lieux | 23–92 % | road-detail fields often left blank | Medium |
| `secu2` | usagers | ≈ 43 % | 2nd safety equipment (often none) | Medium |
| `adr` | caract | ≈ 4 % | free-text address | Low (redundant with lat/long) |
| `an_nais` | usagers | ≈ 2 % | **birth year → age** | **High** (analysis variable) |
| `trajet` | usagers | ≈ 2 % | trip purpose | Medium |

**Most critical:** `an_nais` (blocks the age computation, a major analysis dimension) and, for any geographic cross-tab, the reliability of `lat`/`long`. Conversely, `lartpc`/`occutc`/pedestrian fields are missing **by design**: imputing them would be a mistake.

### Remediation strategies
- **Structural missingness** (`lartpc`, `occutc`, pedestrian fields) → **keep `NaN`**; do not impute. Optionally create a boolean flag (`is_pedestrian`, `is_public_transport`).
- **Missing `an_nais`** → **do not impute a birth year** (would distort age). Create an `age = "Unknown"` category and exclude these rows from age-based analyses.
- **Sentinel `-1`** → systematically recode to `NaN` + label `"Not reported"` in the dimensions (Silver layer).
- **Missing `adr`** → non-blocking: geolocation relies on `lat`/`long`.

In [7]:
usg = raw["usagers"].copy()
usg["an_nais"] = pd.to_numeric(usg["an_nais"], errors="coerce")
usg["age"] = 2024 - usg["an_nais"]

print("an_nais missing / non-numeric :", usg["an_nais"].isna().sum())
print("Negative ages (an_nais > 2024):", (usg["age"] < 0).sum())
print("Ages > 100 years               :", (usg["age"] > 100).sum())
print(f"Observed age range : [{usg['age'].min():.0f} ; {usg['age'].max():.0f}] years")
print("\nDistribution by age band :")
bins = [0,14,17,24,34,44,54,64,74,200]
labels = ["0-14","15-17","18-24","25-34","35-44","45-54","55-64","65-74","75+"]
print(pd.cut(usg["age"], bins=bins, labels=labels).value_counts().sort_index().to_string())

an_nais missing / non-numeric : 2579
Negative ages (an_nais > 2024): 0
Ages > 100 years               : 6
Observed age range : [0 ; 110] years

Distribution by age band :
age
0-14      6482
15-17     4790
18-24    23366
25-34    24269
35-44    19476
45-54    16613
55-64    13116
65-74     7557
75+       6764


In [11]:
print("--- Strictly identical rows (true duplicates) ---")
for name, df in raw.items():
    print(f"{name:10s}: {df.duplicated().sum():>5d} 100% duplicated row(s)")

print("\n--- Uniqueness of the Num_Acc key ---")
for name in ["caract", "lieux"]:
    df = raw[name]
    n_dup_key = df.duplicated(subset=["Num_Acc"]).sum()
    print(f"{name:10s}: {df['Num_Acc'].nunique():,} unique accidents / "
          f"{len(df):,} rows  ->  {n_dup_key:,} repeated key(s)")

print("\n--- Nature of the repetitions in lieux (segments per accident) ---")
print(raw["lieux"]["Num_Acc"].value_counts().value_counts().sort_index()
        .rename_axis("n_lieux_rows").reset_index(name="n_accidents").to_string(index=False))

print("\nInterpretation: repeated Num_Acc in lieux = multi-lane junctions.")
print("Remediation = DO NOT drop_duplicates blindly. Pick 1 main segment per accident")

--- Strictly identical rows (true duplicates) ---
caract    :     0 100% duplicated row(s)
lieux     :     2 100% duplicated row(s)
vehicules :     0 100% duplicated row(s)
usagers   :     0 100% duplicated row(s)

--- Uniqueness of the Num_Acc key ---
caract    : 54,402 unique accidents / 54,402 rows  ->  0 repeated key(s)
lieux     : 54,402 unique accidents / 70,248 rows  ->  15,846 repeated key(s)

--- Nature of the repetitions in lieux (segments per accident) ---
 n_lieux_rows  n_accidents
            1        38757
            2        15463
            3          164
            4           17
            5            1

Interpretation: repeated Num_Acc in lieux = multi-lane junctions.
Remediation = DO NOT drop_duplicates blindly. Pick 1 main segment per accident


In [9]:
geo = raw["caract"].copy()
for col in ["lat", "long"]:
    geo[col] = pd.to_numeric(geo[col].str.replace(",", ".", regex=False), errors="coerce")

print("Non-convertible coordinates (lat) :", geo["lat"].isna().sum())
print("Non-convertible coordinates (long):", geo["long"].isna().sum())
print("Null coordinates (0,0)            :", ((geo["lat"] == 0) & (geo["long"] == 0)).sum())
print(f"lat range  : [{geo['lat'].min():.3f} ; {geo['lat'].max():.3f}]")
print(f"long range : [{geo['long'].min():.3f} ; {geo['long'].max():.3f}]")
metro = geo["lat"].between(41, 51.5) & geo["long"].between(-5.5, 9.8)
print(f"\nAccidents outside mainland bounds : {(~metro).sum():,} "
      f"({(~metro).mean()*100:.2f} %) — mostly overseas territories (valid)")
impossible = (~geo["lat"].between(-90, 90)) | (~geo["long"].between(-180, 180))
print("Geographically impossible coordinates :", impossible.sum())

Non-convertible coordinates (lat) : 0
Non-convertible coordinates (long): 0
Null coordinates (0,0)            : 0
lat range  : [-22.433 ; 51.079]
long range : [-178.094 ; 167.863]

Accidents outside mainland bounds : 3,347 (6.15 %) — mostly overseas territories (valid)
Geographically impossible coordinates : 0


In [13]:
DOMAINS = {
    ("usagers","catu"):   {"1","2","3"},
    ("usagers","sexe"):   {"1","2"},
    ("usagers","grav"):   {"1","2","3","4"},
    ("usagers","trajet"): {"0","1","2","3","4","5","9"},
    ("caract","lum"):     {"1","2","3","4","5"},
    ("caract","agg"):     {"1","2"},
    ("caract","atm"):     {"1","2","3","4","5","6","7","8","9"},
    ("caract","col"):     {"1","2","3","4","5","6","7"},
    ("caract","int"):     {"1","2","3","4","5","6","7","8","9"},
}
print(f"{'table.column':22s} {'out-of-domain':>13s}  {'of which -1 (unknown)':>22s}")
for (tbl,col), valid in DOMAINS.items():
    s = raw[tbl][col].astype("string").str.strip()
    out = s[~s.isin(valid) & s.notna()]
    n_sentinel = (out == "-1").sum()
    n_true = len(out) - n_sentinel
    print(f"{tbl+'.'+col:22s} {n_true:>13d}  {n_sentinel:>22d}")

print("\n=> No value outside its nomenclature: the categoricals are VALID.")
print("   Only defect = sentinel -1 (unknown), to recode as missing value.")

table.column           out-of-domain   of which -1 (unknown)
usagers.catu                       0                       0
usagers.sexe                       0                    2395
usagers.grav                       0                       0
usagers.trajet                     0                    2626
caract.lum                         0                       0
caract.agg                         0                       0
caract.atm                         0                       0
caract.col                         0                       6
caract.int                         0                       0

=> No value outside its nomenclature: the categoricals are VALID.
   Only defect = sentinel -1 (unknown), to recode as missing value.
